# Contour Detection on SPY 2019-2021

Apply the rolling CFAD detector to SPY returns and examine alarms around the February 2020 Flash Crash and March 2020 COVID crash.

In [ ]:
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'cfad').exists():
    ROOT = ROOT.parent
if not (ROOT / 'cfad').exists():
    raise RuntimeError('Unable to locate project root for cfad')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cfad import detect
from cfad.utils import load_spy_sample

## Load and detect

In [ ]:
returns = load_spy_sample()
returns = returns.loc['2019-01-01':'2021-12-31']
report = detect(returns, window=60, h=4.0)
report.summary()

## Score and alarm plot

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True, figsize=(12, 7))
ax[0].plot(returns.index, returns.values, color='tab:blue', label='Returns')
alarm_dates = report.alarm_dates
if alarm_dates is not None and len(alarm_dates):
    ax[0].scatter(alarm_dates, returns.reindex(alarm_dates).values,
                  color='red', marker='x', s=80, label='Alarms')
ax[0].set_title('SPY returns with CFAD alarms')
ax[0].legend()
ax[0].grid(True, alpha=0.3)
ax[1].plot(report.scores, label='Residue score', color='tab:green')
ax[1].axhline(report.threshold, color='red', linestyle='--', label='Threshold')
ax[1].set_title('CFAD anomaly score')
ax[1].set_xlabel('Window index')
ax[1].legend()
ax[1].grid(True, alpha=0.3)
figure_path = Path('paper/figures/03_contour_detection.png')
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout()
fig.savefig(figure_path, dpi=150)
print(f'Saved contour detection figure to {figure_path}')